# TF-IDF baseline for rating bands

The input is `data/tokenized/digikala-comments-tokenized.parquet`. Polars loads it. `rate` becomes `rating_band` and is not a model feature. The only features are the tokenizer ids.

Bands, with ordinal codes `low=0`, `medium=1`, `high=2`:

- `low`: 0 ≤ rate < 2.5
- `medium`: 2.5 ≤ rate < 4.0
- `high`: 4.0 ≤ rate ≤ 5.0

A missing or out-of-range rate is `invalid` and is left out of this classifier. Each product is placed in one split: 70% train, 15% validation, 15% test. Reviews of one product stay together. The TF-IDF vocabulary is fit on the training split only, then a logistic regression predicts `rating_band`.


In [1]:
from pathlib import Path
import hashlib
import sys

import polars as pl
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TOKENIZED = ROOT / "data/tokenized/digikala-comments-tokenized.parquet"
TOKENIZER_NAME = "HooshvareLab/bert-fa-base-uncased"
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15
BANDS = ["low", "medium", "high"]
BAND_CODE = {"low": 0, "medium": 1, "high": 2}
SPLITS = ["train", "validation", "test"]
SPLIT_SHARE = {"train": TRAIN_SHARE, "validation": VALIDATION_SHARE, "test": TEST_SHARE}


/Users/amir/miniconda3/envs/maktab/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Label `rate`

`low` is every rate from 0 up to but not including 2.5, so a rate of 0 stays low. `medium` runs up to but not including 4. `high` includes 5.


In [2]:
def rating_band(rate: pl.Expr) -> pl.Expr:
    """Map a raw rate onto the project bands. Rate 0 stays in low."""
    return (
        pl.when(rate.is_null() | (rate < 0) | (rate > 5))
        .then(pl.lit("invalid"))
        .when(rate < 2.5)
        .then(pl.lit("low"))
        .when(rate < 4.0)
        .then(pl.lit("medium"))
        .otherwise(pl.lit("high"))
    )


reviews = pl.read_parquet(TOKENIZED).with_columns(
    rating_band(pl.col("rate")).alias("rating_band")
).with_columns(
    pl.col("rating_band").replace_strict(BAND_CODE, default=None).alias("rating_band_id")
)

band_counts = (
    reviews.group_by("rating_band")
    .agg(pl.len().alias("rows"))
    .with_columns((pl.col("rows") / reviews.height * 100).round(2).alias("percent"))
    .sort("rating_band")
)
print("rows:", reviews.height)
band_counts


rows: 6152207


rating_band,rows,percent
str,u32,f64
"""high""",4002165,65.05
"""low""",1091653,17.74
"""medium""",1058389,17.2


## Split by product

Products are ordered by a hash of `SPLIT_SEED` and `product_id`, then assigned to the split that is furthest below its target share of each band. A product's reviews all receive that one split. Invalid rows keep the product's split and are not used to train or score this head.


In [3]:
def product_order_key(product_id: int) -> int:
    key = f"{SPLIT_SEED}|{product_id}".encode()
    return int(hashlib.sha256(key).hexdigest()[:16], 16)


labeled = reviews.filter(pl.col("rating_band") != "invalid")
products = (
    labeled.group_by("product_id")
    .agg(
        pl.len().alias("n"),
        (pl.col("rating_band") == "low").sum().alias("n_low"),
        (pl.col("rating_band") == "medium").sum().alias("n_medium"),
        (pl.col("rating_band") == "high").sum().alias("n_high"),
    )
    .sort("product_id")
)
ordered = sorted(
    products.iter_rows(named=True),
    key=lambda row: (product_order_key(row["product_id"]), row["product_id"]),
)
totals = {band: sum(row[f"n_{band}"] for row in ordered) for band in BANDS}
targets = {
    split: {band: SPLIT_SHARE[split] * totals[band] for band in BANDS}
    for split in SPLITS
}
filled = {split: {band: 0 for band in BANDS} for split in SPLITS}


def split_room(split: str, row: dict) -> float:
    score = 0.0
    for band in BANDS:
        target = targets[split][band]
        if target == 0:
            continue
        score += row[f"n_{band}"] * (target - filled[split][band]) / target
    return score


assignments = {}
for row in ordered:
    chosen = max(SPLITS, key=lambda split: split_room(split, row))
    assignments[row["product_id"]] = chosen
    for band in BANDS:
        filled[chosen][band] += row[f"n_{band}"]

split_frame = pl.DataFrame(
    {"product_id": list(assignments), "split": list(assignments.values())}
)
reviews = reviews.join(split_frame, on="product_id", how="left")
labeled = reviews.filter(pl.col("rating_band") != "invalid")

shared_products = (
    labeled.group_by("product_id")
    .agg(pl.col("split").n_unique().alias("n_splits"))
    .filter(pl.col("n_splits") > 1)
    .height
)
print("products in more than one split:", shared_products)

split_counts = (
    labeled.group_by("split", "rating_band")
    .agg(pl.len().alias("rows"))
    .with_columns((pl.col("rows") / labeled.height * 100).round(2).alias("percent_of_labeled"))
    .sort("split", "rating_band")
)
split_counts


products in more than one split: 0


split,rating_band,rows,percent_of_labeled
str,str,u32,f64
"""test""","""high""",600332,9.76
"""test""","""low""",163747,2.66
"""test""","""medium""",158765,2.58
"""train""","""high""",2801508,45.54
"""train""","""low""",764155,12.42
"""train""","""medium""",740859,12.04
"""validation""","""high""",600325,9.76
"""validation""","""low""",163751,2.66
"""validation""","""medium""",158765,2.58


## TF-IDF and logistic regression

Each review is the space-separated tokenizer ids from `input_ids`, without the classifier, separator, or pad ids. `TfidfVectorizer` sees the training documents only. Logistic regression then predicts `low`, `medium`, and `high`. Class weights are balanced from the training rows.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
special_ids = [
    token_id
    for token_id in (tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)
    if token_id is not None
]

documents = labeled.with_columns(
    pl.col("input_ids")
    .list.eval(
        pl.when(pl.element().is_in(special_ids))
        .then(None)
        .otherwise(pl.element().cast(pl.String))
    )
    .list.drop_nulls()
    .list.join(" ")
    .alias("token_text")
).select("id", "product_id", "split", "rating_band", "rating_band_id", "token_text")

train = documents.filter(pl.col("split") == "train")
validation = documents.filter(pl.col("split") == "validation")
test = documents.filter(pl.col("split") == "test")
print("train", train.height, "validation", validation.height, "test", test.height)

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\d+\b", min_df=5)
x_train = vectorizer.fit_transform(train.get_column("token_text").to_list())
y_train = train.get_column("rating_band_id").to_numpy()
print("vocabulary:", len(vectorizer.vocabulary_))

classifier = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=200,
    random_state=SPLIT_SEED,
)
classifier.fit(x_train, y_train)


def score_split(name: str, frame: pl.DataFrame) -> None:
    features = vectorizer.transform(frame.get_column("token_text").to_list())
    actual = frame.get_column("rating_band_id").to_numpy()
    predicted = classifier.predict(features)
    print(name)
    print("accuracy:", round(float(accuracy_score(actual, predicted)), 4))
    print("balanced accuracy:", round(float(balanced_accuracy_score(actual, predicted)), 4))
    print("macro f1:", round(float(f1_score(actual, predicted, average="macro")), 4))
    print("ordinal distance error:", round(float(abs(actual - predicted).mean()), 4))
    print(classification_report(actual, predicted, labels=[0, 1, 2], target_names=BANDS, digits=4))
    print(confusion_matrix(actual, predicted, labels=[0, 1, 2]))
    print()


score_split("validation", validation)
score_split("test", test)


train 4306522 validation 922841 test 922844
vocabulary: 34185
validation
accuracy: 0.6316
balanced accuracy: 0.5705
macro f1: 0.5495
ordinal distance error: 0.487
              precision    recall  f1-score   support

         low     0.4600    0.5143    0.4856    163751
      medium     0.3440    0.4986    0.4071    158765
        high     0.8230    0.6988    0.7558    600325

    accuracy                         0.6316    922841
   macro avg     0.5424    0.5705    0.5495    922841
weighted avg     0.6762    0.6316    0.6479    922841

[[ 84213  35630  43908]
 [ 33329  79153  46283]
 [ 65542 115283 419500]]

test
accuracy: 0.6319
balanced accuracy: 0.5697
macro f1: 0.5492
ordinal distance error: 0.4866
              precision    recall  f1-score   support

         low     0.4608    0.5128    0.4854    163747
      medium     0.3436    0.4959    0.4060    158765
        high     0.8219    0.7003    0.7562    600332

    accuracy                         0.6319    922844
   macro avg  

## Save the fitted baseline

The vectorizer and the classifier go in one file under `models/`. That directory stays out of git. Only `models/.gitkeep` is tracked.

In [6]:
import joblib

model_dir = ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "rating-band-tfidf-logreg.joblib"
joblib.dump(
    {
        "vectorizer": vectorizer,
        "classifier": classifier,
        "bands": BANDS,
        "band_code": BAND_CODE,
        "tokenizer_name": TOKENIZER_NAME,
        "split_seed": SPLIT_SEED,
    },
    model_path,
)
print(model_path)
print("bytes:", model_path.stat().st_size)

/Users/amir/Documents/GitHub/digikala-products-recommender-systems/models/rating-band-tfidf-logreg.joblib
bytes: 1464709
